# 04 — Verify yearly routing destinations

Check the destination products generated by `ANAL/routing/02_build_yearly_destinations.ipynb`. Inputs combine GIP motorway exits, public-transport stops and stations, higher education, and curated centres. Historical OSM POIs are not used.

In [1]:
from pathlib import Path
import warnings

import geopandas as gpd
import pandas as pd

warnings.filterwarnings("ignore", message="Value .* parsed incompletely.*", category=RuntimeWarning)

def discover_project_dir() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "ANAL").is_dir() and (candidate / "OGD").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the project directory or a subdirectory containing ANAL/ and OGD/.")

# Local configuration: change only these paths when transferring the project to the air-gapped PC.
PROJECT_DIR = discover_project_dir()
POI_DIR = PROJECT_DIR / "ANAL" / "data" / "routing" / "destinations"
MUNICIPALITIES_PATH = PROJECT_DIR / "OGD" / "Gemeindegrenzen.zip"
CRS_ANALYSIS = "EPSG:3035"
CRS_ROUTING = "EPSG:4326"
YEARS = range(2015, 2026)
EXPECTED_TYPES = {"motorway_exit", "regional_centre", "urban_centre", "higher_education", "pt_stop"}
REQUIRED_COLUMNS = {"year", "poi_type", "poi_id", "source_file", "source_schema", "provenance", "geometry"}

if not MUNICIPALITIES_PATH.exists():
    raise FileNotFoundError(f"Offline preflight failed; missing municipality boundary archive: {MUNICIPALITIES_PATH}")

In [2]:
municipalities = gpd.read_file(f"zip://{MUNICIPALITIES_PATH.resolve().as_posix()}").to_crs(CRS_ANALYSIS)
study_area = gpd.GeoSeries([municipalities.union_all().buffer(5_000)], crs=CRS_ANALYSIS).to_crs(CRS_ROUTING).iloc[0]
records = []
reference_columns = reference_crs = None
for year in YEARS:
    poi_path = POI_DIR / f"austria-{year}-pois.geoparquet"
    if not poi_path.exists():
        records.append({"year": year, "status": "CHECK", "detail": "missing POI file"})
        continue
    pois = gpd.read_parquet(poi_path)
    columns = tuple(pois.columns)
    crs = pois.crs.to_string() if pois.crs else None
    if reference_columns is None:
        reference_columns, reference_crs = columns, crs
    checks = {
        "required schema": REQUIRED_COLUMNS.issubset(pois.columns),
        "schema consistent": columns == reference_columns,
        "CRS consistent": crs == reference_crs == "EPSG:3035",
        "year consistent": set(pois["year"].dropna().astype(int)) == {year},
        "POI types valid": set(pois["poi_type"].dropna()) == EXPECTED_TYPES,
        "geometry complete": pois.geometry.notna().all(),
        "PT route IDs complete": "pt_route_ids" in pois and pois.loc[pois["poi_type"].eq("pt_stop"), "pt_route_ids"].notna().all(),
    }
    for detail, passed in checks.items():
        records.append({"year": year, "status": "OK" if passed else "CHECK", "detail": detail})

verification = pd.DataFrame(records)
if (verification.status != "OK").any():
    raise AssertionError(verification[verification.status != "OK"].to_string(index=False))
verification.groupby(["detail", "status"]).size().rename("years").reset_index()

,detail,status,years
0,CRS consistent,OK,11
1,POI types valid,OK,11
2,PT route IDs complete,OK,11
3,geometry complete,OK,11
4,required schema,OK,11
5,schema consistent,OK,11
6,year consistent,OK,11


## Public-transport inventory by analysis year


In [3]:
pt_rows = []
for year in YEARS:
    pois = gpd.read_parquet(POI_DIR / f"austria-{year}-pois.geoparquet")
    stops = pois[pois["poi_type"].eq("pt_stop")]
    route_ids = {
        route
        for value in stops["pt_route_ids"].dropna().astype(str)
        for route in value.split("|")
        if route
    }
    pt_rows.append({
        "year": year,
        "PT stops": len(stops),
        "weekday departures": stops["pt_departures_weekday"].sum(),
        "distinct route IDs": len(route_ids),
    })

pd.DataFrame(pt_rows)


,year,PT stops,weekday departures,distinct route IDs
0,2015,7562,225310.0,633
1,2016,7562,225310.0,633
2,2017,7562,225310.0,633
3,2018,7444,217149.5,568
4,2019,7426,234328.5,559
5,2020,7117,238259.0,546
6,2021,7302,253773.5,705
7,2022,7081,255635.0,535
8,2023,7081,255635.0,535
9,2024,7435,255568.5,551
